# Chapter 3: Basic Vector Calculus

Companion notebook to Section 3 of the study guide (Kundu, Cohen & Dowling 2016, Appendix B.3).
Write your own notes under each heading as you study.

Before committing: **Restart** the kernel, then **Run All**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from ch03_tools import (area_integral, boundary_flux_rectangle, circulation_rectangle,
                        cross_index, curl_z_2d, div_2d, erosion_rate_from_hilltop_curvature,
                        exner_rate, flow_divergence, grad_2d, levi_civita,
                        steady_ridge_linear, steady_ridge_nonlinear)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE = "#2a78d6"                           # one series
RAMP = ["#86b6ef", "#2a78d6", "#104281"]   # ordered series: low -> high
GRAY = "#52514e"                           # reference curves and contours
DIVERGING = LinearSegmentedColormap.from_list("conv_div", ["#2a78d6", "#f0efec", "#e34948"])
SEQUENTIAL = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#2a78d6", "#0d366b"])

## 3.1 The operator $\nabla$ and what grad, div and curl mean

- **Gradient** $\nabla\rho$: points up the steepest ascent, normal to the contours; on a DEM, $-\nabla z$ points downslope and $|\nabla z|$ is the slope.
- **Divergence** $\nabla\cdot\mathbf u$: net outflow per unit volume.
- **Curl** $\nabla\times\mathbf u$: circulation per unit area; the vorticity is **twice** the local angular velocity.

*My notes:*

In [ ]:
# (a)-(b) a Gaussian hill and its diffusive soil flux; (c) a Rankine vortex
x = np.linspace(-300, 300, 241)
X, Y = np.meshgrid(x, x)
d = x[1] - x[0]
z = 50 * np.exp(-(X**2 + Y**2) / (2 * 100**2))   # hill 50 m high, about 100 m wide
zx, zy = grad_2d(z, d, d)
kappa = 0.01                                      # soil diffusivity [m^2/yr]
qx, qy = -kappa * zx, -kappa * zy                 # soil flux per unit width [m^2/yr]
div_q = div_2d(qx, qy, d, d) * 1e3                # [mm/yr]

xv = np.linspace(-3, 3, 241)                      # vortex: core radius 1, Omega = 1
XV, YV = np.meshgrid(xv, xv)
R2 = XV**2 + YV**2
rate = np.where(R2 < 1, 1.0, 1.0 / np.maximum(R2, 1.0))   # u_theta / r
u, v = -rate * YV, rate * XV
omega = curl_z_2d(u, v, xv[1] - xv[0], xv[1] - xv[0])

s = slice(10, None, 20)                           # thin out the arrows
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
ax = axes[0]
ax.contour(X, Y, z, levels=np.arange(5, 50, 5), colors=GRAY, linewidths=0.6, linestyles="solid")
ax.quiver(X[s, s], Y[s, s], -zx[s, s], -zy[s, s], color=BLUE)
ax.set(title="(a) grad: -grad z points downslope,\nacross the contours",
       xlabel="x (m)", ylabel="y (m)")

ax = axes[1]
lim = np.abs(div_q).max()
im = ax.pcolormesh(X, Y, div_q, cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim), shading="auto")
ax.quiver(X[s, s], Y[s, s], qx[s, s], qy[s, s], color=GRAY)
fig.colorbar(im, ax=ax, label="div q (mm/yr)")
ax.set(title="(b) div q: red diverges (erodes),\nblue converges (fills)", xlabel="x (m)")

ax = axes[2]
im = ax.pcolormesh(XV, YV, omega, cmap=SEQUENTIAL, shading="auto")
ax.streamplot(xv, xv, u, v, color=GRAY, density=0.7, linewidth=0.7, arrowsize=0.8)
fig.colorbar(im, ax=ax, label="vorticity (1/s)")
ax.set(title="(c) curl: circular paths everywhere,\nrotation only inside the core",
       xlabel="x / core radius", ylabel="y / core radius")
for ax in axes:
    ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch03_grad_div_curl.png", dpi=150)
plt.show()
print(f"hilltop: div q = {div_q.max():.3f} mm/yr; vortex core: vorticity = {omega[120, 120]:.2f} 1/s")

### $\nabla$ is an operator, not a vector

$(\mathbf u\cdot\nabla)\mathbf v = u_j\,\partial v_i/\partial x_j$ is **not** $\mathbf v\,(\nabla\cdot\mathbf u) = v_i\,\partial u_j/\partial x_j$: $\nabla$ acts on what stands to its right.

*My notes:*

In [ ]:
x, y, z = sp.symbols("x y z", real=True)
X = (x, y, z)
u = sp.Matrix([x, 0, 0])
v = sp.Matrix([y, x, 0])
u_grad_v = sp.Matrix([sum(u[j] * sp.diff(v[i], X[j]) for j in range(3)) for i in range(3)])
v_div_u = v * sum(sp.diff(u[j], X[j]) for j in range(3))
print("(u . grad) v =", list(u_grad_v))   # [0, x, 0]
print("v (div u)    =", list(v_div_u))    # [y, x, 0]

## 3.2 Index notation: the fastest way to prove identities

Summation convention; $\delta_{ij}$ substitutes indices; $(\mathbf a\times\mathbf b)_i = \varepsilon_{ijk}a_jb_k$; and the $\varepsilon$-$\delta$ identity
$\varepsilon_{ijk}\varepsilon_{ilm} = \delta_{jl}\delta_{km} - \delta_{jm}\delta_{kl}$.

Dictionary: $\nabla\phi\to\partial_i\phi$, $\nabla\cdot\mathbf u\to\partial_iu_i$, $\nabla\times\mathbf u\to\varepsilon_{ijk}\partial_ju_k$, $\mathbf u\cdot\nabla\to u_j\partial_j$, $\nabla^2\to\partial_j\partial_j$.

*My notes:*

In [ ]:
eps, delta = levi_civita(), np.eye(3)
lhs = np.einsum("ijk,ilm->jklm", eps, eps)
rhs = np.einsum("jl,km->jklm", delta, delta) - np.einsum("jm,kl->jklm", delta, delta)
print("epsilon-delta identity holds:", np.allclose(lhs, rhs))          # True

a, b = np.array([1.0, 2.0, 3.0]), np.array([-2.0, 0.5, 4.0])
print("a x b, index notation:", cross_index(a, b), "| numpy:", np.cross(a, b))

S = np.array([[2.0, 1.0, 0.0], [1.0, 3.0, 5.0], [0.0, 5.0, -1.0]])   # symmetric
print("epsilon_ijk S_jk =", np.einsum("ijk,jk->i", eps, S))           # zero vector
print("delta_ii =", np.trace(delta))                                  # 3

## 3.3 The identities at a glance

(B.3.1)-(B.3.13). Three consequences used again and again: the Lamb form $(\mathbf u\cdot\nabla)\mathbf u = \nabla(\tfrac12|\mathbf u|^2) - \mathbf u\times\boldsymbol\omega$, the two forms of continuity, and the viscous term $\nabla^2\mathbf u = -\nabla\times\boldsymbol\omega$ for incompressible flow.

*My notes:*

In [ ]:
X = x, y, z = sp.symbols("x y z", real=True)
u = sp.Matrix([sp.Function(n)(*X) for n in ("u1", "u2", "u3")])   # a general vector field
p = sp.Function("p")(*X)                                            # a general scalar field

grad = lambda f: sp.Matrix([sp.diff(f, s) for s in X])
div = lambda F: sum(sp.diff(F[i], X[i]) for i in range(3))
curl = lambda F: sp.Matrix([sp.diff(F[2], y) - sp.diff(F[1], z),
                            sp.diff(F[0], z) - sp.diff(F[2], x),
                            sp.diff(F[1], x) - sp.diff(F[0], y)])
lap = lambda F: F.applyfunc(lambda c: sum(sp.diff(c, s, 2) for s in X))   # Cartesian only
adv = lambda F, G: sp.Matrix([sum(F[j] * sp.diff(G[i], X[j]) for j in range(3))
                              for i in range(3)])                          # (F . grad) G

print("(B.3.11)", list(curl(grad(p))))                                            # [0, 0, 0]
print("(B.3.12)", sp.simplify(div(curl(u))))                                      # 0
print("(B.3.13)", list((curl(curl(u)) - grad(div(u)) + lap(u)).applyfunc(sp.simplify)))
r = sp.sqrt(x**2 + y**2 + z**2)
print("(B.3.3) ", sp.simplify(div(sp.Matrix(X) / r**3)))                          # 0, r != 0
lamb = adv(u, u) - grad(u.dot(u) / 2) + u.cross(curl(u))
print("Lamb form", list(lamb.applyfunc(sp.simplify)))                             # [0, 0, 0]

## 3.4 Integral theorems

Gauss: $\displaystyle\int_A \mathbf u\cdot\mathbf n\,dA = \int_V \nabla\cdot\mathbf u\,dV$. Stokes: $\displaystyle\oint_C \mathbf u\cdot\mathbf t\,ds = \int_A (\nabla\times\mathbf u)\cdot\mathbf n\,dA$.

Localisation: if $\int_V F\,dV = 0$ for **every** volume and $F$ is continuous, then $F = 0$ everywhere. This turns a budget into a PDE.

*My notes:*

In [ ]:
x = np.linspace(0, 1, 401)
y = np.linspace(0, 2, 801)
X, Y = np.meshgrid(x, y)
dx, dy = x[1] - x[0], y[1] - y[0]

qx, qy = X**2 * Y, X * Y**3   # Gauss: flux out of the rectangle vs. integral of div q
print(f"Gauss:  boundary flux = {boundary_flux_rectangle(qx, qy, x, y):.4f}"
      f" | integral of div q = {area_integral(div_2d(qx, qy, dx, dy), x, y):.4f}")   # 6, 6

u, v = -Y**3, X**3            # Stokes: circulation vs. integral of the vorticity
print(f"Stokes: circulation   = {circulation_rectangle(u, v, x, y):.4f}"
      f" | integral of curl_z  = {area_integral(curl_z_2d(u, v, dx, dy), x, y):.4f}")   # 10, 10

## 3.5 Research connection: landscapes as the divergence of fluxes

- **Exner:** $(1-\lambda_p)\,\partial z_b/\partial t = -\nabla_h\cdot\mathbf q_s$ (+ sources): the bed rises where fluxes converge.
- **Hillslopes:** $\mathbf q_s = -\kappa\nabla z$ gives $\partial z/\partial t = U + \kappa\nabla^2 z$; at steady state $\nabla^2 z = -U/\kappa$, so hilltop curvature records the erosion rate. Nonlinear law: $\mathbf q_s = -\kappa\nabla z/[1-(|\nabla z|/S_c)^2]$.
- **Landscape evolution:** $\partial z/\partial t = U - KA^m|\nabla z|^n + \kappa\nabla^2 z$.
- **Flow routing:** $\hat{\mathbf n} = -\nabla z/|\nabla z|$; $\nabla\cdot\hat{\mathbf n} > 0$ on noses and ridges, $< 0$ in hollows.
- **Groundwater:** $\mathbf q = -K_h\nabla h$ with $\nabla\cdot\mathbf q = 0$.

*My notes:*

In [ ]:
# Exner: bedload flux grows by 1e-6 m^2/s over a 1 km reach, bed porosity 0.4
rate = exner_rate(div_qs=1e-6 / 1000, porosity=0.4)                      # [m/s]
print(f"bed change: {rate * 3.156e7 * 100:+.1f} cm/yr (negative = degradation)")   # -5.3

In [ ]:
kappa, S_c, L_H = 0.005, 1.25, 50.0   # soil diffusivity [m^2/yr], critical gradient, half-width [m]
x = np.linspace(0, L_H, 501)
U = 2e-4                                # uplift = erosion rate at steady state: 0.2 mm/yr

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(x, steady_ridge_linear(x, U, kappa), "--", color=GRAY, label="linear flux law")
axes[0].plot(x, steady_ridge_nonlinear(x, U, kappa, S_c), color=BLUE, label="nonlinear flux law")
axes[0].set(xlabel="distance from crest (m)", ylabel="elevation below crest (m)",
            title="(a) same uplift rate (0.2 mm/yr), two flux laws")
axes[0].legend(loc="lower left", fontsize=8)
for U_mm, color, style in zip([0.05, 0.2, 0.5], RAMP, ["-", "--", "-."]):
    profile = steady_ridge_nonlinear(x, U_mm * 1e-3, kappa, S_c)
    axes[1].plot(x, -np.gradient(profile, x), style, color=color, label=f"U = {U_mm} mm/yr")
axes[1].axhline(S_c, color=GRAY, ls=":", lw=1.5)
axes[1].text(49, S_c + 0.02, "critical gradient S_c", ha="right", va="bottom", fontsize=8, color=GRAY)
axes[1].set(ylim=(0, 1.4), xlabel="distance from crest (m)", ylabel="hillslope gradient (-)",
            title="(b) nonlinear law: faster uplift, gradients near S_c")
axes[1].legend(loc="upper left", bbox_to_anchor=(0, 0.86), fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch03_hillslope.png", dpi=200)
plt.show()

# Hilltop curvature records the erosion rate for both flux laws
xs = np.linspace(-1, 1, 201)            # 1 m on either side of the crest
for name, profile in [("linear", steady_ridge_linear(xs, U, kappa)),
                      ("nonlinear", steady_ridge_nonlinear(np.abs(xs), U, kappa, S_c))]:
    C_ht = np.gradient(np.gradient(profile, xs), xs)[100]
    E = erosion_rate_from_hilltop_curvature(C_ht, kappa)
    print(f"{name:9s}: hilltop curvature {C_ht:.3f} 1/m -> E = {E * 1e3:.2f} mm/yr")   # 0.20

In [ ]:
# Flow routing: unit downslope direction and its divergence on a ridge-and-hollow slope
S0, A, wavelength = 0.2, 5.0, 50.0       # regional gradient [-], ridge amplitude [m], spacing [m]
x = np.linspace(0, 2 * wavelength, 201)
y = np.linspace(0, 60, 121)
X, Y = np.meshgrid(x, y)
z = S0 * Y + A * np.cos(2 * np.pi * X / wavelength)   # ridges at x = 0, 50, 100 m
dx, dy = x[1] - x[0], y[1] - y[0]
dn = flow_divergence(z, dx, dy)
zx, zy = grad_2d(z, dx, dy)
g = np.hypot(zx, zy)

fig, ax = plt.subplots(figsize=(8, 4.8))
lim = np.abs(dn).max()
im = ax.pcolormesh(X, Y, dn, cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim), shading="auto")
ax.contour(X, Y, z, levels=15, colors=GRAY, linewidths=0.6, linestyles="solid")
s = (slice(6, None, 12), slice(3, None, 8))
ax.quiver(X[s], Y[s], -zx[s] / g[s], -zy[s] / g[s], color="#0b0b0b", width=0.003)
fig.colorbar(im, ax=ax, label="div of flow direction (1/m)")
ax.set(xlabel="x (m)", ylabel="y (m), uphill",
       title="Flow lines spread on ridges (red) and converge in hollows (blue)")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch03_flow_routing.png", dpi=150)
plt.show()

## Exercises 3.1-3.8

*Work space: solve each exercise here, then check against Appendix C of the study guide.*